# 9주차 코드 필사 — YOLOv8을 활용한 포즈 추정 (Pose Estimation)

---

## 모델 개요: YOLOv8

**YOLO(You Only Look Once)** 는 이미지 전체를 한 번에 처리하는 단일 단계(one-stage) 객체 탐지 모델이다.  
기존의 두 단계(two-stage) 모델(예: Faster R-CNN)과 달리, 별도의 후보 영역 추출 없이 한 번의 순전파로 바운딩 박스와 클래스를 동시에 예측하므로 처리 속도가 매우 빠르다.

**YOLOv8**은 YOLOv5의 구조를 개선한 모델로, 다음과 같은 특징을 갖는다:

| 특징 | 설명 |
|------|------|
| 앵커 프리(Anchor-free) | YOLOv6처럼 앵커 없이 추론하여 속도 향상 |
| 모자이크 합성 제한 | 마지막 10 에폭에서만 모자이크 적용 → 과대적합 방지 |
| 다중 작업 지원 | 검출(Detection), 세그멘테이션(Segmentation), 분류(Classification), **포즈 추정(Pose Estimation)** |
| 사전 학습 데이터 | MS COCO (80개 클래스) |

**포즈 추정(Pose Estimation)** 은 이미지나 비디오에서 사람 또는 동물의 관절 위치와 연결 구조를 파악해 신체 자세를 분석·추론하는 기술이다.  
운동 분석, 로봇 공학, 보안 시스템 등 다양한 분야에 활용된다.

---

## 1. 라이브러리 설치

Ultralytics 라이브러리는 YOLOv8의 공식 구현체로, **PyTorch**와 **OpenCV**를 기반으로 동작한다.  
OpenCV(Open Source Computer Vision Library)는 실시간 영상 처리에 특화된 오픈소스 컴퓨터비전 라이브러리다.

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
pip install ultralytics opencv-python

## 2. YOLOv8 포즈 추정 모델 불러오기 (예제 9.25)

`YOLO` 클래스를 통해 사전 학습된 `yolov8m-pose` 모델을 불러온다.  
지원되는 YOLOv8 모델 크기는 **n, s, m, l, x** 총 다섯 종류이며, 접미사(suffix)로 작업 유형을 지정한다:

- 접미사 없음: 기본 검출(Detection) 모델
- `-seg`: 세그멘테이션 모델
- `-cls`: 분류 모델
- **`-pose`: 포즈 추정 모델** ← 이번 실습에서 사용

포즈 추정 모델은 MS COCO 데이터셋 기준으로 **사람 객체만** 예측한다.

In [21]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/yolov8m-pose.pt")

## 3. 비디오 파일 불러오기 (예제 9.26)

OpenCV의 `VideoCapture` 클래스를 사용해 비디오 파일을 읽는다.

- **VideoCapture**: 문자열 입력 시 파일 경로로 인식, 정수(0, 1 등) 입력 시 카메라 장치 번호로 인식
- **waitKey**: 키 이벤트 발생까지 대기하거나 지정 시간(ms) 대기. 현재 예제는 키 입력 없으면 10ms 대기 후 반복
- **get / set**: 비디오의 속성(현재 프레임, 총 프레임 수 등)을 읽거나 변경
  - `CAP_PROP_POS_FRAMES`: 현재 프레임 위치
  - `CAP_PROP_FRAME_COUNT`: 전체 프레임 수
- **read**: 프레임을 읽어 `(ret, frame)` 반환. `ret=True`이면 정상 읽기, `frame`은 numpy 배열
- **imshow**: 이미지를 별도 창에 출력 (Google Colab에서는 `cv2_imshow` 사용)
- **release / destroyAllWindows**: 메모리 해제 및 모든 창 제거

In [22]:
import cv2
# from google.colab.patches import cv2_imshow  # 출력 저장 용량 초과 방지를 위해 비활성화

# 비디오 열기
capture = cv2.VideoCapture('/content/drive/MyDrive/cat walking.mp4')

# 영상 열기 확인
if not capture.isOpened():
    print("영상 파일을 열 수 없습니다.")
else:
    print("영상 재생 시작")

while True:

    # 프레임 읽기
    ret, frame = capture.read()

    # 프레임 읽기 실패 시 종료
    if not ret:
        print("영상 종료")
        break

    # 프레임 출력
    # cv2_imshow(frame)  # 출력 저장 용량 초과 방지를 위해 비활성화

# 자원 해제
capture.release()
# cv2.destroyAllWindows()  # Colab에서 불필요하여 비활성화

영상 재생 시작
영상 종료


## 4. 모델 추론 함수 정의 (예제 9.27)

`predict` 함수는 YOLOv8 모델의 예측을 수행한다. 주요 파라미터:

| 파라미터 | 설명 |
|----------|------|
| `source` | 추론할 이미지 또는 프레임 |
| `device` | GPU 인덱스 번호 또는 `"cpu"` |
| `iou` | 중복 경계 상자 제거 임계값 (너무 높으면 중복 상자가 남음) |
| `conf` | 클래스 점수 임계값 (설정값보다 낮은 예측 제거) |
| `verbose` | 모델 수행 시 로그 출력 여부 |

YOLOv8은 배치 형태로 이미지를 입력받으므로, 단일 프레임 처리 시 `results[0]`만 사용한다.

In [23]:
import torch

def predict(frame, iou=0.7, conf=0.25):
    results = model(
        source=frame,
        device="0" if torch.cuda.is_available() else "cpu",
        iou=0.7,
        conf=0.25,
        verbose=False,
    )
    result = results[0]
    return result

## 5. 경계 상자 시각화 함수 정의 (예제 9.28)

`results[0].boxes`의 `data` 속성은 `[x1, y1, x2, y2, conf, cls]` 구조로 출력된다.

- **신뢰도(conf)**: 추론된 객체의 점수
- **클래스(cls)**: 객체의 인덱스 번호 (YOLOv8 예측 결과의 `names`와 매핑)

OpenCV의 `rectangle` 함수로 경계 상자를 그린다.  
입력 파라미터: **이미지, (x1, y1), (x2, y2), 색상(BGR), 선 두께**  
두께를 음수로 입력하면 내부가 채워진 사각형이 그려진다.

In [24]:
def draw_boxes(result, frame):
    for boxes in result.boxes:
        x1, y1, x2, y2, score, classes = boxes.data.squeeze().cpu().numpy()
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 1)
    return frame

## 6. 키 포인트 시각화 함수 정의 (예제 9.30)

포즈 추정 모델의 키 포인트 속성을 추출해 시각화한다.

- **Annotator**: Ultralytics 라이브러리의 시각화 클래스. 이미지와 선 두께를 전달해 인스턴스 생성
- **kpts 메서드**: `[17, 3]` 형태의 데이터를 입력받음. MS COCO 키 포인트 데이터셋은 **17개 신체 부위**를 예측하며, 각 포인트는 `[x, y, conf]` 구조
- 정확도가 **0.5 이상**인 키 포인트만 시각화 (모든 포인트를 보려면 `nkps[:,2] = 1`로 설정)
- `circle`: 이미지, 중심점, 반지름, 색상, 두께를 입력받아 원을 그림
- `putText`: 이미지, 문자열, 위치, 글꼴, 글꼴 크기, 색상, 두께를 입력받아 텍스트를 그림

**MS COCO 17개 키 포인트 의미:**

| ID | 의미 | ID | 의미 |
|----|------|----|------|
| 0 | 코 | 9 | 왼쪽 손목 |
| 1 | 왼쪽 눈 | 10 | 오른쪽 손목 |
| 2 | 오른쪽 눈 | 11 | 왼쪽 골반 |
| 3 | 왼쪽 귀 | 12 | 오른쪽 골반 |
| 4 | 오른쪽 귀 | 13 | 왼쪽 무릎 |
| 5 | 왼쪽 어깨 | 14 | 오른쪽 무릎 |
| 6 | 오른쪽 어깨 | 15 | 왼쪽 발목 |
| 7 | 왼쪽 팔꿈치 | 16 | 오른쪽 발목 |
| 8 | 오른쪽 팔꿈치 | | |

In [25]:
!pip install ultralytics

In [26]:
from ultralytics.utils.plotting import Annotator

def draw_keypoints(result, frame):
    annotator = Annotator(frame, line_width=1)
    for kps in result.keypoints:
        kps = kps.data.squeeze()
        annotator.kpts(kps)

        nkps = kps.cpu().numpy()
        # nkps[:,2] = 1
        # annotator.kpts(nkps)
        for idx, (x, y, score) in enumerate(nkps):
            if score > 0.5:
                cv2.circle(frame, (int(x), int(y)), 3, (0, 0, 255), cv2.FILLED)
                cv2.putText(
                    frame, str(idx), (int(x), int(y)), cv2.FONT_HERSHEY_COMPLEX,
                    1, (0, 0, 255), 1
                )

    return frame

## 7. 모델 추론 및 시각화 적용 (예제 9.29 + 키 포인트 통합)

앞서 정의한 `predict`, `draw_boxes`, `draw_keypoints` 함수를 while 루프에 통합 적용한다.  
각 프레임마다 모델 추론 → 경계 상자 시각화 → 키 포인트 시각화 순으로 수행한다.

**YOLOv8의 장점 요약:**
- 이미지 전체를 한 번에 처리 → 다른 딥러닝 모델 대비 빠른 처리 속도
- 다양한 크기와 객체들의 겹침 같은 복잡한 상황에서도 우수한 성능
- 최신 딥러닝 기술과 작업별 모델 제공 → 구현이 간편함

In [27]:
from google.colab.patches import cv2_imshow
from IPython.display import clear_output

capture = cv2.VideoCapture("/content/drive/MyDrive/cat walking.mp4")

# 저장 설정
fps = capture.get(cv2.CAP_PROP_FPS)
width  = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("/content/drive/MyDrive/cat_result.mp4", fourcc, fps, (width, height))

while True:
    if capture.get(cv2.CAP_PROP_POS_FRAMES) == capture.get(cv2.CAP_PROP_FRAME_COUNT):
        break

    ret, frame = capture.read()
    if not ret:
        break

    result = predict(frame)

    frame = draw_boxes(result, frame)
    frame = draw_keypoints(result, frame)

    out.write(frame)
    # clear_output(wait=True)  # 출력 저장 용량 초과 방지를 위해 비활성화
    # cv2_imshow(frame)        # 출력 저장 용량 초과 방지를 위해 비활성화

capture.release()
out.release()
print("저장 완료!")


저장 완료!
